# NESA — Medical Knowledge Base: Embedding Generation & Indexing

**Scope of this notebook:** ONLY embedding generation and indexing (Dense Vector Index + Sparse BM25 Index + Metadata Store).

This notebook does **NOT** implement:
- Retrieval pipelines (query routing, hybrid fusion, RRF, etc.)
- Reranking
- Generation / LLM answer synthesis

Those are handled in downstream notebooks that will consume the artifacts produced here:
- A **persisted vector collection** (ChromaDB, local disk)
- A **persisted BM25 index** (`data/processed/indices/bm25_index.pkl`)

## Why Hybrid Indexing (Dense + BM25) for a Dual-Persona Medical RAG system?

NESA serves two personas with very different query styles, and neither retrieval method alone covers both well:

- **Postpartum Assistant (lay users):** Queries are conversational and paraphrased
  ("my baby won't stop crying at night, is that normal postpartum?"). This favors
  **dense embeddings**, which capture semantic/contextual similarity even when exact
  wording differs from the source documents.
- **Research Assistant (clinicians/researchers):** Queries often contain precise,
  non-negotiable terminology — drug names, dosages, guideline names, ICD codes,
  acronyms ("WHO Labour Care Guide", "MgSO4 4g IV loading dose", "ACOG Level II NICU criteria").
  Dense embeddings can blur these fine-grained lexical distinctions, so an exact-match
  **BM25 sparse index** is essential to guarantee these terms are not missed.
- **Metadata filtering** (`document_type`, `section`, etc.) lets us further scope retrieval
  per persona/role (e.g. restrict the Postpartum Assistant to `patient_education` /
  `book` content, and give the Research Assistant access to `clinical_guideline` /
  `research_paper` content), reducing the risk of surfacing clinically inappropriate
  content to the wrong audience.

Combining both signals (to be fused downstream, e.g. via Reciprocal Rank Fusion) gives
robustness: BM25 acts as a safety net for exact medical terms/dosages that dense
retrieval might semantically "smooth over," while dense embeddings handle paraphrase
and conceptual queries that pure keyword matching would miss entirely.


## 1. Setup & Configuration

All paths are relative to the project root (`data/processed/...`), so this notebook can
run from any machine/CI job as long as it's executed from the repo root.

Two embedding backends are supported and selected via `EMBEDDING_BACKEND`:
- `"openai"` — `text-embedding-3-small` / `text-embedding-3-large` (requires `OPENAI_API_KEY`)
- `"sentence_transformers"` — local `BAAI/bge-large-en-v1.5` (no API key, runs on CPU/GPU)


In [2]:
from pathlib import Path
from dataclasses import dataclass
from typing import Literal

# --------------------------------------------------------------------------
# Path configuration
# --------------------------------------------------------------------------

CHUNKS_PATH = Path(
    r"D:\new\OneDrive\Desktop\cleaned_Data\chunking\nesa_chunks.jsonl"
)

INDICES_DIR = Path(
    r"D:\new\OneDrive\Desktop\cleaned_Data\INDICES"
)

VECTOR_DB_DIR = Path(
    r"D:\new\OneDrive\Desktop\cleaned_Data\VECTOR_DB"
)

BM25_INDEX_PATH = INDICES_DIR / "bm25_index.pkl"

# Create output directories if they don't exist
INDICES_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)


@dataclass
class NesaConfig:
    """Central configuration for the embedding + indexing pipeline."""

    # --- Embedding backend -------------------------------------------------
    embedding_backend: Literal["openai", "sentence_transformers"] = (
        "sentence_transformers"
    )

    # OpenAI options
    openai_model: str = "text-embedding-3-small"
    openai_dimensions: int | None = None

    # Local SentenceTransformers model
    st_model_name: str = "BAAI/bge-large-en-v1.5"

    # --- Batch processing --------------------------------------------------
    batch_size: int = 64
    max_retries: int = 5
    initial_backoff_seconds: float = 1.0
    backoff_multiplier: float = 2.0

    # --- Vector DB ---------------------------------------------------------
    vector_db_backend: Literal["chromadb"] = "chromadb"
    collection_name: str = "nesa_medical_kb"

    # --- Text fields -------------------------------------------------------
    text_field_for_embedding: str = "text"
    text_field_for_bm25: str = "text"

    # --- BM25 --------------------------------------------------------------
    bm25_k1: float = 1.5
    bm25_b: float = 0.75


CONFIG = NesaConfig()

print("Configuration loaded:")
for k, v in CONFIG.__dict__.items():
    print(f"  {k}: {v}")

print(f"\nChunks input path : {CHUNKS_PATH}")
print(f"Indices output dir : {INDICES_DIR}")
print(f"Vector DB output dir : {VECTOR_DB_DIR}")
print(f"BM25 index path : {BM25_INDEX_PATH}")

Configuration loaded:
  embedding_backend: sentence_transformers
  openai_model: text-embedding-3-small
  openai_dimensions: None
  st_model_name: BAAI/bge-large-en-v1.5
  batch_size: 64
  max_retries: 5
  initial_backoff_seconds: 1.0
  backoff_multiplier: 2.0
  vector_db_backend: chromadb
  collection_name: nesa_medical_kb
  text_field_for_embedding: text
  text_field_for_bm25: text
  bm25_k1: 1.5
  bm25_b: 0.75

Chunks input path : D:\new\OneDrive\Desktop\cleaned_Data\chunking\nesa_chunks.jsonl
Indices output dir : D:\new\OneDrive\Desktop\cleaned_Data\INDICES
Vector DB output dir : D:\new\OneDrive\Desktop\cleaned_Data\VECTOR_DB
BM25 index path : D:\new\OneDrive\Desktop\cleaned_Data\INDICES\bm25_index.pkl


## 2. Imports & Client Initializations

We lazily initialize only the embedding client that matches `CONFIG.embedding_backend`,
so the notebook doesn't require both `openai` and `sentence-transformers` to be installed.


In [3]:
import os
import re
import json
import time
import pickle
import logging
from typing import Any, Iterable, Iterator

import numpy as np
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("nesa_indexing")

# --- Vector DB: ChromaDB ----------------------------------------------------
import chromadb
from chromadb.config import Settings

# --- Sparse index: BM25 -----------------------------------------------------
from rank_bm25 import BM25Okapi

# --- Embedding backends (loaded conditionally) ------------------------------
openai_client = None
st_model = None

if CONFIG.embedding_backend == "openai":
    from openai import OpenAI

    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        raise EnvironmentError(
            "OPENAI_API_KEY not set. Export it, or switch "
            "CONFIG.embedding_backend to 'sentence_transformers' for a local fallback."
        )
    openai_client = OpenAI(api_key=api_key)
    logger.info(f"Initialized OpenAI client with model={CONFIG.openai_model}")

elif CONFIG.embedding_backend == "sentence_transformers":
    from sentence_transformers import SentenceTransformer

    st_model = SentenceTransformer(CONFIG.st_model_name)
    logger.info(f"Loaded SentenceTransformer model={CONFIG.st_model_name}")

else:
    raise ValueError(f"Unknown embedding_backend: {CONFIG.embedding_backend}")

# --- Chroma client (persistent, local disk) ---------------------------------
chroma_client = chromadb.PersistentClient(
    path=str(VECTOR_DB_DIR),
    settings=Settings(anonymized_telemetry=False),
)
logger.info(f"Initialized ChromaDB PersistentClient at {VECTOR_DB_DIR}")


c:\Users\E_Magic\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-18 15:10:23,072 | INFO | No device provided, using cpu
2026-08-18 15:10:23,879 | INFO | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-08-18 15:10:24,060 | INFO | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-08-18 15:10:24,061 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-08-18 15:10:24,100 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-08-18 15:10:2

## 3. Data Ingestion

Load the chunked JSONL produced by the chunking stage, validate record counts, and
inspect the token-count distribution to sanity-check the chunking (e.g. flag
unusually small/large chunks before we spend money/time embedding them).


In [4]:
def load_chunks(jsonl_path: Path) -> list[dict[str, Any]]:
    """Load chunk records from a JSONL file.

    Args:
        jsonl_path: Path to the newline-delimited JSON file of chunks.

    Returns:
        A list of chunk dicts, in file order.

    Raises:
        FileNotFoundError: If the JSONL file does not exist.
        ValueError: If no valid records are found.
    """
    if not jsonl_path.exists():
        raise FileNotFoundError(f"Chunks file not found: {jsonl_path}")

    records: list[dict[str, Any]] = []
    malformed = 0
    with jsonl_path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                malformed += 1
                logger.warning(f"Skipping malformed JSON on line {line_no}: {e}")

    if not records:
        raise ValueError(f"No valid chunk records found in {jsonl_path}")

    if malformed:
        logger.warning(f"Skipped {malformed} malformed line(s) while loading {jsonl_path}")

    return records


def validate_chunks(chunks: list[dict[str, Any]]) -> dict[str, Any]:
    """Validate chunk structure and compute summary statistics.

    Handles missing/null metadata fields gracefully rather than raising.

    Returns:
        A stats dict with counts, token distribution, and missing-field report.
    """
    required_fields = ["chunk_id", "document_id", "text"]
    missing_field_counts: dict[str, int] = {f: 0 for f in ["title", "chapter", "section", "document_type"]}
    token_counts: list[int] = []
    seen_ids: set[str] = set()
    duplicate_ids = 0

    for c in chunks:
        for rf in required_fields:
            if not c.get(rf):
                raise ValueError(f"Chunk missing required field '{rf}': {c.get('chunk_id', '<unknown>')}")

        for field_name in missing_field_counts:
            if c.get(field_name) is None:
                missing_field_counts[field_name] += 1

        tc = c.get("token_count")
        if isinstance(tc, (int, float)):
            token_counts.append(int(tc))

        cid = c["chunk_id"]
        if cid in seen_ids:
            duplicate_ids += 1
        seen_ids.add(cid)

    stats = {
        "total_chunks": len(chunks),
        "unique_chunk_ids": len(seen_ids),
        "duplicate_chunk_ids": duplicate_ids,
        "missing_field_counts": missing_field_counts,
        "token_count_min": int(np.min(token_counts)) if token_counts else None,
        "token_count_max": int(np.max(token_counts)) if token_counts else None,
        "token_count_mean": float(np.mean(token_counts)) if token_counts else None,
        "token_count_median": float(np.median(token_counts)) if token_counts else None,
        "document_type_distribution": {},
    }

    doc_types: dict[str, int] = {}
    for c in chunks:
        dt = c.get("document_type") or "unknown"
        doc_types[dt] = doc_types.get(dt, 0) + 1
    stats["document_type_distribution"] = dict(sorted(doc_types.items(), key=lambda kv: -kv[1]))

    return stats


chunks = load_chunks(CHUNKS_PATH)
ingestion_stats = validate_chunks(chunks)

print(f"Loaded {ingestion_stats['total_chunks']} chunks "
      f"({ingestion_stats['unique_chunk_ids']} unique IDs, "
      f"{ingestion_stats['duplicate_chunk_ids']} duplicates)")
print(f"Token count — min: {ingestion_stats['token_count_min']}, "
      f"max: {ingestion_stats['token_count_max']}, "
      f"mean: {ingestion_stats['token_count_mean']:.1f}, "
      f"median: {ingestion_stats['token_count_median']:.1f}")
print("\nDocument type distribution:")
for dt, count in ingestion_stats["document_type_distribution"].items():
    print(f"  {dt}: {count}")


Loaded 13107 chunks (13107 unique IDs, 0 duplicates)
Token count — min: 35, max: 20632, mean: 260.5, median: 179.0

Document type distribution:
  book: 13107


## 4. Embedding Engine & Batch Generator

Embeddings are generated on the **enriched `text` field** (contextual headers +
content) rather than `raw_text`, per the design spec — this gives the embedding
model document-type/section/page context that improves semantic retrieval quality.

Batches of `CONFIG.batch_size` are embedded per API/model call, with exponential
backoff retry logic to handle rate limits gracefully.


In [ ]:
def batched(iterable: list[Any], batch_size: int) -> Iterator[list[Any]]:
    """Yield successive batches of `batch_size` from `iterable`."""
    for i in range(0, len(iterable), batch_size):
        yield iterable[i : i + batch_size]


def embed_batch_openai(texts: list[str], config: NesaConfig) -> list[list[float]]:
    """Embed a batch of texts using the OpenAI embeddings API with retry/backoff.

    Args:
        texts: Batch of input strings to embed.
        config: Pipeline configuration (model name, retry settings).

    Returns:
        A list of embedding vectors, one per input text, in the same order.

    Raises:
        RuntimeError: If all retry attempts are exhausted.
    """
    backoff = config.initial_backoff_seconds
    kwargs: dict[str, Any] = {"model": config.openai_model, "input": texts}
    if config.openai_dimensions:
        kwargs["dimensions"] = config.openai_dimensions

    for attempt in range(1, config.max_retries + 1):
        try:
            response = openai_client.embeddings.create(**kwargs)
            # OpenAI preserves input order in `response.data`
            return [item.embedding for item in response.data]
        except Exception as e:  # broad on purpose: covers RateLimitError, APIError, timeouts, etc.
            if attempt == config.max_retries:
                raise RuntimeError(f"Embedding batch failed after {attempt} attempts: {e}") from e
            logger.warning(f"Embedding attempt {attempt}/{config.max_retries} failed ({e}); "
                            f"retrying in {backoff:.1f}s")
            time.sleep(backoff)
            backoff *= config.backoff_multiplier

    raise RuntimeError("Unreachable: exhausted retries without raising")


def embed_batch_sentence_transformers(texts: list[str], config: NesaConfig) -> list[list[float]]:
    """Embed a batch of texts using a local SentenceTransformers model.

    Args:
        texts: Batch of input strings to embed.
        config: Pipeline configuration (unused here beyond interface symmetry).

    Returns:
        A list of embedding vectors, one per input text, in the same order.
    """
    vectors = st_model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
    return [v.tolist() for v in vectors]


def embed_texts(texts: list[str], config: NesaConfig) -> list[list[float]]:
    """Dispatch to the configured embedding backend for a single batch."""
    if config.embedding_backend == "openai":
        return embed_batch_openai(texts, config)
    elif config.embedding_backend == "sentence_transformers":
        return embed_batch_sentence_transformers(texts, config)
    raise ValueError(f"Unknown embedding_backend: {config.embedding_backend}")


def generate_embeddings_for_chunks(
    chunks: list[dict[str, Any]], config: NesaConfig
) -> tuple[list[dict[str, Any]], list[list[float]]]:
    """Generate embeddings for all chunks in batches.

    Args:
        chunks: List of chunk dicts (must contain `config.text_field_for_embedding`).
        config: Pipeline configuration.

    Returns:
        Tuple of (chunks, embeddings) where embeddings[i] corresponds to chunks[i].
    """
    all_embeddings: list[list[float]] = []
    text_field = config.text_field_for_embedding

    batches = list(batched(chunks, config.batch_size))
    for batch in tqdm(batches, desc="Embedding batches", unit="batch"):
        texts = [c.get(text_field) or c.get("raw_text") or "" for c in batch]
        embeddings = embed_texts(texts, config)
        all_embeddings.extend(embeddings)

    assert len(all_embeddings) == len(chunks), (
        f"Embedding count mismatch: {len(all_embeddings)} embeddings for {len(chunks)} chunks"
    )
    return chunks, all_embeddings


embedding_start_time = time.time()
chunks, embeddings = generate_embeddings_for_chunks(chunks, CONFIG)
embedding_elapsed = time.time() - embedding_start_time

embedding_dim = len(embeddings[0]) if embeddings else 0
print(f"Generated {len(embeddings)} embeddings (dim={embedding_dim}) in {embedding_elapsed:.1f}s")


Embedding batches:  14%|█▎        | 28/205 [49:06<5:16:36, 107.33s/batch]

## 5. Vector DB Indexing (ChromaDB)

We create (or reset) a persistent Chroma collection and insert vectors with a full
metadata payload for role-based filtering downstream. Chroma metadata values must be
`str`, `int`, `float`, or `bool` — `None` values are coerced to safe defaults so
inserts don't fail on partially-populated chunk metadata.


In [ ]:
def sanitize_metadata(chunk: dict[str, Any]) -> dict[str, Any]:
    """Build a Chroma-safe metadata payload from a chunk record.

    Chroma metadata does not accept None / list / dict values, so we:
      - coerce None -> "" (strings) or -1 (ints) as appropriate
      - JSON-encode list fields (e.g. source_block_indices) to strings

    Args:
        chunk: Raw chunk dict.

    Returns:
        A flat dict of Chroma-compatible metadata values.
    """
    def s(value: Any, default: str = "") -> str:
        return value if isinstance(value, str) and value else default

    def i(value: Any, default: int = -1) -> int:
        return int(value) if isinstance(value, (int, float)) else default

    return {
        "chunk_id": s(chunk.get("chunk_id")),
        "document_id": s(chunk.get("document_id")),
        "document_type": s(chunk.get("document_type"), default="unknown"),
        "section": s(chunk.get("section"), default="unknown"),
        "title": s(chunk.get("title"), default="untitled"),
        "chapter": s(chunk.get("chapter"), default="none"),
        "page_start": i(chunk.get("page_start")),
        "page_end": i(chunk.get("page_end")),
        "content_type": s(chunk.get("content_type"), default="text"),
        "token_count": i(chunk.get("token_count")),
        "raw_text": s(chunk.get("raw_text")),
        "text": s(chunk.get("text")),
        "source_file": s(chunk.get("source_file")),
    }


def index_chunks_to_vector_db(
    chunks: list[dict[str, Any]],
    embeddings: list[list[float]],
    client: "chromadb.PersistentClient",
    config: NesaConfig,
    reset_collection: bool = True,
) -> "chromadb.Collection":
    """Create a Chroma collection and insert all chunk embeddings + metadata.

    Args:
        chunks: Chunk records aligned index-for-index with `embeddings`.
        embeddings: Dense embedding vectors aligned index-for-index with `chunks`.
        client: An initialized Chroma PersistentClient.
        config: Pipeline configuration (collection name, batch size).
        reset_collection: If True, delete any existing collection with the same
            name before creating a fresh one (idempotent re-runs).

    Returns:
        The populated Chroma collection.
    """
    if reset_collection:
        try:
            client.delete_collection(config.collection_name)
            logger.info(f"Deleted existing collection '{config.collection_name}'")
        except Exception:
            pass  # collection did not exist yet

    collection = client.get_or_create_collection(
        name=config.collection_name,
        metadata={"hnsw:space": "cosine"},
    )

    for batch_chunks, batch_embeds in zip(
        tqdm(list(batched(chunks, config.batch_size)), desc="Indexing to Chroma", unit="batch"),
        batched(embeddings, config.batch_size),
    ):
        ids = [c["chunk_id"] for c in batch_chunks]
        metadatas = [sanitize_metadata(c) for c in batch_chunks]
        documents = [c.get("text") or c.get("raw_text") or "" for c in batch_chunks]

        collection.upsert(
            ids=ids,
            embeddings=batch_embeds,
            metadatas=metadatas,
            documents=documents,
        )

    return collection


vector_indexing_start = time.time()
collection = index_chunks_to_vector_db(chunks, embeddings, chroma_client, CONFIG)
vector_indexing_elapsed = time.time() - vector_indexing_start

record_count = collection.count()
print(f"Vector DB collection '{CONFIG.collection_name}' now has {record_count} records "
      f"(expected {len(chunks)}) — indexed in {vector_indexing_elapsed:.1f}s")
assert record_count == len(chunks), "Vector DB record count does not match chunk count!"


## 6. BM25 Sparse Index Creation & Persistence

Tokenization lowercases and strips punctuation, but **deliberately preserves tokens
containing digits** (e.g. `4g`, `500mg`, `37.5c`) since dosages/units are exactly the
kind of exact-match terms BM25 needs to catch for the Research Assistant use case.
We apply light stemming (Porter stemmer via NLTK) to normalize morphological variants
(e.g. "monitoring"/"monitor") while leaving numeric/unit tokens untouched.


In [ ]:
try:
    import nltk
    from nltk.stem import PorterStemmer

    for resource in ["punkt", "punkt_tab"]:
        try:
            nltk.data.find(f"tokenizers/{resource}")
        except LookupError:
            nltk.download(resource, quiet=True)

    _stemmer = PorterStemmer()
    _STEMMING_AVAILABLE = True
except Exception as e:
    logger.warning(f"NLTK stemming unavailable ({e}); falling back to unstemmed tokens.")
    _STEMMING_AVAILABLE = False

# Standard English stopwords MINUS terms that matter in medical/dosage context
# (e.g. we keep "per", "of" out of the removal list would be too aggressive; instead
# we simply do not remove numeric/unit-bearing tokens regardless of stopword status).
_STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "is", "are", "was", "were", "be", "been",
    "being", "in", "on", "at", "to", "for", "with", "as", "by", "this", "that",
    "these", "those", "it", "its", "from", "which", "who", "whom",
}

_TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:[.\-/][a-z0-9]+)*")  # keeps 500mg, mg/kg, 37.5c, etc.
_HAS_DIGIT = re.compile(r"\d")


def tokenize_medical_text(text: str) -> list[str]:
    """Tokenize text for BM25 indexing, preserving dosage/unit tokens.

    Steps: lowercase -> pattern-based tokenization (keeps embedded digits/units) ->
    stopword removal (numeric tokens are exempt) -> light stemming (numeric tokens exempt).

    Args:
        text: Raw or enriched chunk text.

    Returns:
        List of normalized tokens.
    """
    if not text:
        return []

    lowered = text.lower()
    raw_tokens = _TOKEN_PATTERN.findall(lowered)

    tokens: list[str] = []
    for tok in raw_tokens:
        has_digit = bool(_HAS_DIGIT.search(tok))
        if not has_digit and tok in _STOPWORDS:
            continue
        if _STEMMING_AVAILABLE and not has_digit:
            tok = _stemmer.stem(tok)
        tokens.append(tok)

    return tokens


def build_bm25_index(
    chunks: list[dict[str, Any]], config: NesaConfig
) -> tuple["BM25Okapi", list[str]]:
    """Build a BM25Okapi index over the corpus.

    Args:
        chunks: Chunk records.
        config: Pipeline configuration (text field, k1/b hyperparameters).

    Returns:
        Tuple of (bm25_index, doc_id_mapping) where doc_id_mapping[i] is the
        chunk_id corresponding to the i-th document in the BM25 corpus.
    """
    text_field = config.text_field_for_bm25
    corpus_texts = [c.get(text_field) or c.get("raw_text") or "" for c in chunks]
    tokenized_corpus = [tokenize_medical_text(t) for t in tqdm(corpus_texts, desc="Tokenizing for BM25")]

    bm25 = BM25Okapi(tokenized_corpus, k1=config.bm25_k1, b=config.bm25_b)
    doc_id_mapping = [c["chunk_id"] for c in chunks]

    return bm25, doc_id_mapping


def save_bm25_index(
    bm25: "BM25Okapi",
    doc_id_mapping: list[str],
    chunks: list[dict[str, Any]],
    path: Path,
) -> None:
    """Persist the BM25 index, doc-id mapping, and chunk lookup to disk.

    Args:
        bm25: Fitted BM25Okapi index.
        doc_id_mapping: doc_id_mapping[i] -> chunk_id for the i-th BM25 document.
        chunks: Full chunk records, saved for metadata lookup at query time.
        path: Output pickle path (parent dirs must already exist).
    """
    chunk_lookup = {c["chunk_id"]: c for c in chunks}
    payload = {
        "bm25_index": bm25,
        "doc_id_mapping": doc_id_mapping,
        "chunk_lookup": chunk_lookup,
        "tokenizer_uses_stemming": _STEMMING_AVAILABLE,
    }
    with path.open("wb") as f:
        pickle.dump(payload, f)


def load_bm25_index(path: Path) -> dict[str, Any]:
    """Load a previously persisted BM25 index payload from disk."""
    with path.open("rb") as f:
        return pickle.load(f)


bm25_start = time.time()
bm25_index, bm25_doc_id_mapping = build_bm25_index(chunks, CONFIG)
save_bm25_index(bm25_index, bm25_doc_id_mapping, chunks, BM25_INDEX_PATH)
bm25_elapsed = time.time() - bm25_start

# Verify round-trip load
_reloaded = load_bm25_index(BM25_INDEX_PATH)
assert len(_reloaded["doc_id_mapping"]) == len(chunks), "BM25 reload doc count mismatch!"

print(f"BM25 index built over {len(chunks)} documents in {bm25_elapsed:.1f}s")
print(f"Saved to: {BM25_INDEX_PATH}")
print(f"Reload verification passed ({len(_reloaded['doc_id_mapping'])} docs).")


## 7. Retrieval Verification & Metadata Filtering Test

Lightweight test cells only — this is **not** the retrieval pipeline itself, just a
smoke test proving both indices are queryable and metadata filtering works, per persona.


In [ ]:
def vector_search(
    query: str,
    top_k: int = 5,
    where: dict[str, Any] | None = None,
) -> list[dict[str, Any]]:
    """Run a dense vector similarity search against the Chroma collection.

    Args:
        query: Natural language query string.
        top_k: Number of results to return.
        where: Optional Chroma metadata filter, e.g. {"document_type": "patient_education"}.

    Returns:
        List of result dicts with chunk_id, score (cosine similarity), and metadata.
    """
    query_embedding = embed_texts([query], CONFIG)[0]

    kwargs: dict[str, Any] = {
        "query_embeddings": [query_embedding],
        "n_results": top_k,
        "include": ["metadatas", "documents", "distances"],
    }
    if where:
        kwargs["where"] = where

    results = collection.query(**kwargs)

    output = []
    ids = results["ids"][0]
    distances = results["distances"][0]
    metadatas = results["metadatas"][0]
    for doc_id, distance, metadata in zip(ids, distances, metadatas):
        # Chroma with hnsw:space="cosine" returns cosine DISTANCE; similarity = 1 - distance
        similarity = 1.0 - distance
        output.append({
            "chunk_id": doc_id,
            "score": round(similarity, 4),
            "document_type": metadata.get("document_type"),
            "section": metadata.get("section"),
            "page_start": metadata.get("page_start"),
            "preview": (metadata.get("raw_text") or "")[:160],
        })
    return output


def bm25_search(query: str, top_k: int = 5) -> list[dict[str, Any]]:
    """Run a BM25 keyword search against the persisted BM25 index.

    Args:
        query: Natural language / keyword query string.
        top_k: Number of results to return.

    Returns:
        List of result dicts with chunk_id, BM25 score, and metadata preview.
    """
    tokenized_query = tokenize_medical_text(query)
    scores = bm25_index.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]

    output = []
    for idx in top_indices:
        chunk_id = bm25_doc_id_mapping[idx]
        chunk = next((c for c in chunks if c["chunk_id"] == chunk_id), None)
        output.append({
            "chunk_id": chunk_id,
            "score": round(float(scores[idx]), 4),
            "document_type": chunk.get("document_type") if chunk else None,
            "section": chunk.get("section") if chunk else None,
            "page_start": chunk.get("page_start") if chunk else None,
            "preview": (chunk.get("raw_text") or "")[:160] if chunk else "",
        })
    return output


def print_results(title: str, results: list[dict[str, Any]]) -> None:
    print(f"\n--- {title} ---")
    if not results:
        print("  (no results)")
        return
    for rank, r in enumerate(results, start=1):
        print(f"  {rank}. [{r['score']}] {r['chunk_id']} "
              f"(type={r['document_type']}, section={r['section']}, page={r['page_start']})")
        print(f"       {r['preview']}...")


In [ ]:
# --- Test Query 1: Postpartum Assistant use case ---------------------------
query_1 = "What are the warning signs of postpartum depression?"

dense_results_1 = vector_search(query_1, top_k=5)
print_results(f"Dense Vector Search — '{query_1}'", dense_results_1)

bm25_results_1 = bm25_search(query_1, top_k=5)
print_results(f"BM25 Search — '{query_1}'", bm25_results_1)


In [ ]:
# --- Test Query 2: Research Assistant use case ------------------------------
query_2 = "WHO Labour Care Guide guidelines for stage 1 labor monitoring."

dense_results_2 = vector_search(query_2, top_k=5)
print_results(f"Dense Vector Search — '{query_2}'", dense_results_2)

bm25_results_2 = bm25_search(query_2, top_k=5)
print_results(f"BM25 Search — '{query_2}'", bm25_results_2)


In [ ]:
# --- Filtered Vector Search: role-based metadata filtering ------------------
# Example: restrict results to `document_type == "patient_education"`, simulating
# how the Postpartum Assistant persona would scope retrieval away from
# clinician-oriented research content.
filtered_results = vector_search(
    query_1,
    top_k=5,
    where={"document_type": "patient_education"},
)
print_results(
    f"Filtered Dense Vector Search (document_type='patient_education') — '{query_1}'",
    filtered_results,
)

if not filtered_results:
    print("Note: no chunks matched document_type='patient_education' in this corpus. "
          "Check ingestion_stats['document_type_distribution'] above for available values "
          "and adjust the filter accordingly.")


## 8. Summary & Statistics

Final readiness report for the downstream Hybrid Retrieval & Reranking stage.


In [ ]:
total_pipeline_elapsed = embedding_elapsed + vector_indexing_elapsed + bm25_elapsed

summary = {
    "total_chunks_processed": len(chunks),
    "embedding_backend": CONFIG.embedding_backend,
    "embedding_model": (
        CONFIG.openai_model if CONFIG.embedding_backend == "openai" else CONFIG.st_model_name
    ),
    "embedding_dimension": embedding_dim,
    "vector_db_backend": "ChromaDB (PersistentClient)",
    "vector_db_path": str(VECTOR_DB_DIR),
    "vector_db_collection": CONFIG.collection_name,
    "vector_db_record_count": collection.count(),
    "bm25_index_path": str(BM25_INDEX_PATH),
    "bm25_document_count": len(bm25_doc_id_mapping),
    "timing_seconds": {
        "embedding_generation": round(embedding_elapsed, 2),
        "vector_db_indexing": round(vector_indexing_elapsed, 2),
        "bm25_indexing": round(bm25_elapsed, 2),
        "total": round(total_pipeline_elapsed, 2),
    },
    "downstream_readiness": {
        "hybrid_retrieval_ready": (
            collection.count() == len(chunks) and len(bm25_doc_id_mapping) == len(chunks)
        ),
        "reranking_ready": False,  # implemented in a downstream notebook
        "generation_ready": False,  # implemented in a downstream notebook
    },
}

print(json.dumps(summary, indent=2))

if summary["downstream_readiness"]["hybrid_retrieval_ready"]:
    print("\n✅ Vector index and BM25 index are consistent and ready for Hybrid Retrieval & Reranking.")
else:
    print("\n⚠️  Index counts are inconsistent — investigate before proceeding to Hybrid Retrieval.")
